# 🚲 Bike Demand Forecasting
## Step 4 & 5 — Exploratory Data Analysis & Time Series Readiness

Visualise trends, seasonality, and anomalies. Check stationarity (ADF test). Convert data into a properly indexed, resampled time series ready for modelling.

### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import warnings, os

warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid", palette="muted")
os.makedirs("output_plots", exist_ok=True)

day_df  = pd.read_csv("day.csv")
hour_df = pd.read_csv("hour.csv")
print("Data loaded.")

---
## Step 5 — Time Series Readiness
Convert `dteday` to datetime, build a full datetime index for hourly data, enforce daily frequency, and demonstrate resampling.

In [ ]:
# Convert date column
day_df['dteday']  = pd.to_datetime(day_df['dteday'])
hour_df['dteday'] = pd.to_datetime(hour_df['dteday'])

# Build proper hourly datetime
hour_df['datetime'] = hour_df['dteday'] + pd.to_timedelta(hour_df['hr'], unit='h')

# Set indexes
day_df.set_index('dteday', inplace=True)
hour_df.set_index('datetime', inplace=True)

# Enforce daily frequency (fills any gaps with NaN)
day_df = day_df.asfreq('D')

# Fill any NaN created by asfreq
if day_df['cnt'].isnull().sum() > 0:
    day_df['cnt'] = day_df['cnt'].ffill()

print("Date range (daily) :", day_df.index.min(), "→", day_df.index.max())
print("Date range (hourly):", hour_df.index.min(), "→", hour_df.index.max())

#### Resampling demo — Hourly → Weekly

In [ ]:
weekly = hour_df['cnt'].resample('W').sum()
print(f"Weekly resampled: {weekly.shape[0]} rows")
weekly.tail()

---
## Step 4 — Exploratory Data Analysis

### 4.1 — Overall Daily Trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(day_df.index, day_df['cnt'], color='#60a5fa', linewidth=1.5, label='Daily Rentals')
ax.fill_between(day_df.index, day_df['cnt'], alpha=0.15, color='#60a5fa')
ax.set_title('Daily Bike Rentals — 2011–2012', fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Total Rentals')
ax.legend()
plt.tight_layout()
plt.savefig('output_plots/1_daily_trend.png', dpi=150)
plt.show()

### 4.2 — Hourly Seasonality (Box Plot)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(x='hr', y='cnt', data=hour_df.reset_index(), ax=ax, color='#818cf8')
ax.set_title('Hourly Bike Rental Patterns', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of the Day'); ax.set_ylabel('Rentals')
plt.tight_layout()
plt.savefig('output_plots/2_hourly_seasonality.png', dpi=150)
plt.show()

### 4.3 — Weekly Patterns (Working Day vs Weekend)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(x='weekday', y='cnt', hue='workingday', data=day_df, ax=ax)
ax.set_title('Rentals by Day of Week  (0=Sun, 6=Sat)', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week'); ax.set_ylabel('Rentals')
ax.legend(title='Working Day')
plt.tight_layout()
plt.savefig('output_plots/3_weekly_patterns.png', dpi=150)
plt.show()

### 4.4 — Seasonal Distribution

In [ ]:
season_labels = {1:'Spring', 2:'Summer', 3:'Fall', 4:'Winter'}
df_plot = day_df.copy()
df_plot['Season'] = df_plot['season'].map(season_labels)

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(x='Season', y='cnt', data=df_plot,
            order=['Spring','Summer','Fall','Winter'], ax=ax,
            palette=['#34d399','#60a5fa','#f97316','#a78bfa'])
ax.set_title('Demand Distribution by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output_plots/4_seasonal_distribution.png', dpi=150)
plt.show()

### 4.5 — Stationarity Check (ADF Test)
> **H₀**: The series has a unit root (non-stationary).  
> **Reject H₀** if p-value ≤ 0.05  →  series is stationary.

In [ ]:
adf_stat, p_val, _, _, crit_vals, _ = adfuller(day_df['cnt'])

print(f"ADF Statistic : {adf_stat:.4f}")
print(f"p-value       : {p_val:.6f}")
for k, v in crit_vals.items():
    print(f"  Critical ({k}): {v:.4f}")

if p_val <= 0.05:
    print("\n✅ Data is STATIONARY — safe to apply forecasting models directly.")
else:
    print("\n⚠️  Data is NON-STATIONARY — consider differencing before modelling.")

### Save Ready-to-Model Data

In [ ]:
day_df.to_pickle("day_ready.pkl")
hour_df.to_pickle("hour_ready.pkl")
print("Saved: day_ready.pkl  |  hour_ready.pkl")